本脚本用于处理 PostgreSQL 导出的数据 
便于导入neo4j


### 先处理 两个node


In [2]:
import pandas as pd

In [ ]:
## 读取 重命名 保存
df = pd.read_csv("../Data/PGSQL/actors.csv")
df.rename(columns={"actor_id": "actor_id:ID(Actor)"}, inplace=True)
df.to_csv("../Data/Neo4j/actors_neo4j_node.csv", index=False)


In [7]:
# 加载原始 CSV
df = pd.read_csv("../Data/PGSQL/repos.csv")
df.rename(columns={"repo_id": "repo_id:ID(Repo)"}, inplace=True)
df.to_csv("../Data/Neo4j/repos_neo4j_node_full.csv", index=False)
# columns_to_keep = ["repo_id:ID(Repo)", "repo_name","upstream_marks", "created_at", "indexed"]
# df_cleaned = df[columns_to_keep]
# df_cleaned.to_csv("../Data/Neo4j/repos_neo4j_node.csv", index=False)

### 处理边

In [8]:
df = pd.read_csv("../Data/PGSQL/events.csv")

In [15]:
#  重命名两个关键 ID 字段
df.rename(columns={
    "actor_id": ":START_ID(Actor)",
    "repo_id": ":END_ID(Repo)",
    "id": "event_id",
}, inplace=True)

# 添加 Neo4j 关系类型字段
df[":TYPE"] = "INTERACTS_WITH"

# 设定核心字段顺序
core_fields = [":START_ID(Actor)", ":END_ID(Repo)", ":TYPE"]

# 获取当前列名顺序
all_columns = df.columns.tolist()

# 剔除核心字段，保留其余字段顺序
remaining_columns = [col for col in all_columns if col not in core_fields]

# 构造新的列顺序：先核心字段，再剩余字段
new_column_order = core_fields + remaining_columns

# 重新排列列顺序
df = df[new_column_order]

# 导出为 CSV
df.to_csv("../Data/Neo4j/event_repo_edges_final.csv", index=False)